# Observed vs. predicted fit figures

Builds the two supplement figures from the BEST Tobit fits:

1. **Full grid**: 18 panels, every antibody-antigen complex, titled with the selected
   interaction order and the regularization behind that fit.
2. **Selected sites**: observed vs. predicted for one complex, split by germline or
   mutant at a chosen site. Works for any antibody, antigen, and site.

Observed values are replicate averages per genotype from `{antibody}_kds.csv`.
Predictions are `predicted_kd_full` from `{antibody}_{antigen}_BEST_df_all.csv`.
Both are censored at the 6.0 detection floor.

## Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D

data_dir = "../data"
fig_dir = "../figures"
os.makedirs(fig_dir, exist_ok=True)
png_dpi = 300

plt.rcParams["svg.fonttype"] = "path"
plt.rcParams["pdf.fonttype"] = 42

antigen_hex = {"CH505TF": "#E66100", "SF162": "#5D3A9B"}
germ_hex = "#386A4F"
mut_hex = "#D48DA6"

c_ink = "#1a1a2e"
c_muted = "#6b6b7b"
c_line = "#e3e3ea"
c_spine = "#333333"
c_identity = "#444444"

floor = 6.0         # Tobit detection floor in -log10 Kd units
jitter_band = 0.15  # censored points spread into a band below the floor
jitter_seed = 0
tick_step = 0.5     # shared x and y tick spacing

xlabel = r"Measured Binding Affinity ($-\log_{10}\mathrm{K_D}$)"
ylabel = r"Inferred Binding Affinity ($-\log_{10}\mathrm{K_D}$)"

included_abs = ["CH235.12", "1NC9", "N49P7", "VRC01", "VRC07", "VRC13"]
excluded_abs = ["NIH45-46", "VRC-PG20", "VRC16"]
grid_abs = included_abs + excluded_abs

stem_map = {"CH235.12": "CH235_12", "1NC9": "1NC9", "N49P7": "N49P7", "VRC01": "VRC01",
            "VRC07": "VRC07", "VRC13": "VRC13", "NIH45-46": "NIH4546",
            "VRC-PG20": "VRCPG20", "VRC16": "VRC16"}

site_cols = [f"site_{i}" for i in range(10)]

### Regularization behind each best fit

In [ ]:
alphas = {
    ("CH235.12", "CH505TF"): (0.1, 0.1),  ("CH235.12", "SF162"): (0.1, 0.1),
    ("1NC9", "CH505TF"): (0.1, 0),        ("1NC9", "SF162"): (0.1, 0),
    ("N49P7", "CH505TF"): (0.1, 1),       ("N49P7", "SF162"): (0.1, 0),
    ("VRC01", "CH505TF"): (0.1, 0),       ("VRC01", "SF162"): (0.1, 0),
    ("VRC07", "CH505TF"): (0.1, 0),       ("VRC07", "SF162"): (0.1, 0),
    ("VRC13", "CH505TF"): (1, 0),         ("VRC13", "SF162"): (1, 0.001),
    ("NIH45-46", "CH505TF"): (0.1, 0),    ("NIH45-46", "SF162"): (0.1, 0),
    ("VRC-PG20", "CH505TF"): (0.1, 1),    ("VRC-PG20", "SF162"): (1, 0),
    ("VRC16", "CH505TF"): (0.1, 0.01),    ("VRC16", "SF162"): (1, 0.001),
}

orders_from_tab = {
    ("CH235.12", "CH505TF"): 3, ("CH235.12", "SF162"): 2,
    ("1NC9", "CH505TF"): 4,     ("1NC9", "SF162"): 3,
    ("N49P7", "CH505TF"): 2,    ("N49P7", "SF162"): 5,
    ("VRC01", "CH505TF"): 5,    ("VRC01", "SF162"): 2,
    ("VRC07", "CH505TF"): 4,    ("VRC07", "SF162"): 4,
    ("VRC13", "CH505TF"): 2,    ("VRC13", "SF162"): 2,
    ("NIH45-46", "CH505TF"): 5, ("NIH45-46", "SF162"): 4,
    ("VRC-PG20", "CH505TF"): 2, ("VRC-PG20", "SF162"): 2,
    ("VRC16", "CH505TF"): 5,    ("VRC16", "SF162"): 2,
}

# panels making up the selected-sites supplement figure
site_specs = [("NIH45-46", "SF162", "N52K"),
              ("VRC-PG20", "CH505TF", "S54W"),
              ("VRC16", "SF162", "R83T")]

## Loading and stats

In [ ]:
# Observed, predicted, and selected order for one antibody-antigen complex

def load_complex(ab, ag):
    s = stem_map[ab]
    kds = pd.read_csv(f"{data_dir}/{s}_kds.csv")
    sub = kds[kds["antigen"] == ag].copy()
    sub["tok"] = sub[site_cols].agg("".join, axis=1)  # G/M string, site_0 leftmost
    obs = sub.groupby("tok")["nlog10_Kd"].mean().dropna()

    dfa = pd.read_csv(f"{data_dir}/{s}_{ag}_BEST_df_all.csv")
    dfa["tok"] = dfa[site_cols].agg("".join, axis=1)
    pred = dfa.set_index("tok")["predicted_kd_full"].reindex(obs.index)

    order = int(pd.read_csv(f"{data_dir}/{s}_{ag}_BEST_df_params.csv")["order"].max())
    return dict(geno=obs.index.to_numpy(), obs=obs.to_numpy(),
                pred=pred.to_numpy(), order=order)


# Mutation labels per site index, for example N52K at index 4
def site_labels(ab):
    meta = pd.read_csv(f"{data_dir}/{stem_map[ab]}_meta.csv").reset_index(drop=True)
    return (meta["germline_aa"].astype(str) + meta["kabat_position"].astype(str)
            + meta["mature_aa"].astype(str)).tolist()

# Accept a site as an index 0 to 9 or as a label like N52K
def site_index(ab, site):
    if isinstance(site, (int, np.integer)):
        return int(site)
    labels = site_labels(ab)
    if site not in labels:
        raise ValueError(f"{site} not a site of {ab}. options: {labels}")
    return labels.index(site)


def fit_stats(obs, pred):
    return np.corrcoef(obs, pred)[0, 1], float(np.sqrt(np.mean((obs - pred) ** 2)))


def alpha_line(ab, ag):
    a = alphas.get((ab, ag))
    return None if a is None else f"L1 = {a[0]:g}, L2 = {a[1]:g}"

# Spread floor-censored points below the threshold so the pile-up is readable
def jitter_censored(obs, pred, rng):
    x, y = obs.copy(), pred.copy()
    xf, yf = obs == floor, pred == floor
    x[xf] = floor - rng.uniform(0, jitter_band, int(xf.sum()))
    y[yf] = floor - rng.uniform(0, jitter_band, int(yf.sum()))
    return x, y

## Shared styling

In [ ]:
def shared_range(complexes):
    vals = np.concatenate([np.concatenate([c["obs"], c["pred"]]) for c in complexes])
    lo, hi = np.nanmin(vals), np.nanmax(vals)
    pad = 0.04 * (hi - lo)
    return lo - pad, hi + pad


def style_axes(ax, lo, hi):
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    ax.plot([lo, hi], [lo, hi], ls="--", lw=0.8, color=c_identity, zorder=1)
    for s in ax.spines.values():
        s.set_color(c_spine)
        s.set_linewidth(1.0)
    ticks = np.arange(np.ceil(lo / tick_step) * tick_step, hi + 1e-9, tick_step)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    ax.tick_params(length=3, labelsize=7, colors=c_muted, labelcolor=c_ink)
    ax.grid(True, color="#f0f0f4", lw=0.6, zorder=0)


def stat_box(ax, lines):
    ax.text(0.04, 0.96, "\n".join(lines), transform=ax.transAxes, va="top", ha="left",
            fontsize=6.2, color=c_ink, linespacing=1.35,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=c_line, lw=0.6, alpha=0.9))


def group_header(fig, axes, text, dy=0.032):
    """Center an antibody label above the pair of antigen panels below it."""
    pos = [a.get_position() for a in axes]
    xc = (min(p.x0 for p in pos) + max(p.x1 for p in pos)) / 2
    fig.text(xc, max(p.y1 for p in pos) + dy, text, ha="center", va="bottom",
             fontsize=10.5, color=c_ink, weight="bold")


def save_figure(fig, name):
    """Write png, pdf, and svg, then confirm the svg carries no live text."""
    for ext in ("png", "pdf", "svg"):
        fig.savefig(f"{fig_dir}/{name}.{ext}", dpi=png_dpi, bbox_inches="tight")
    svg = open(f"{fig_dir}/{name}.svg").read()
    assert svg.count("<text") == 0, "live text in svg, set svg.fonttype='path'"
    print(name, "saved (png, pdf, svg)")

## Figure 1: full grid

In [ ]:
def build_grid():
    """18 panel observed vs predicted grid, one antibody pair per column block."""
    cells = [c for ab in grid_abs for c in ((ab, "CH505TF"), (ab, "SF162"))]
    data = [load_complex(ab, ag) for ab, ag in cells]
    lo, hi = shared_range(data)

    fig = plt.figure(figsize=(13.5, 8.1))
    gs = gridspec.GridSpec(3, 6, figure=fig, hspace=0.82, wspace=0.30,
                           left=0.085, right=0.985, top=0.79, bottom=0.11)
    axes = []
    for cell, (ab, ag) in enumerate(cells):
        row, col = divmod(cell, 6)
        ax = fig.add_subplot(gs[row, col])
        axes.append(ax)
        d = data[cell]
        style_axes(ax, lo, hi)
        ax.scatter(d["obs"], d["pred"], s=7, c=antigen_hex[ag], alpha=0.5,
                   linewidths=0, zorder=2)
        r, rmse = fit_stats(d["obs"], d["pred"])
        stat_box(ax, [f"r={r:.3f}", f"RMSE={rmse:.3f}", f"n={len(d['obs'])}"])

        title = f"best-fit order: {d['order']}"
        params = alpha_line(ab, ag)
        if params:
            title += f"\n{params}"
        ax.set_title(title, fontsize=8.5, color=c_ink, pad=4)

    # a params line makes each title two lines, so the antibody label has to clear more
    header_dy = 0.055 if any(alpha_line(ab, ag) for ab, ag in cells) else 0.032
    for j, ab in enumerate(grid_abs):
        group_header(fig, [axes[2 * j], axes[2 * j + 1]], ab, dy=header_dy)

    fig.text(0.5, 0.965, "Observed vs. predicted affinity for best model fits",
             ha="center", fontsize=14, color=c_ink, weight="bold")
    fig.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=antigen_hex[a], mec="none",
                               ms=8, label=a) for a in ("CH505TF", "SF162")],
               loc="upper center", ncol=2, frameon=False, fontsize=10,
               bbox_to_anchor=(0.5, 0.945), columnspacing=1.6, handletextpad=0.4)

    for j in range(3):
        pair = [axes[12 + 2 * j].get_position(), axes[13 + 2 * j].get_position()]
        xc = (min(p.x0 for p in pair) + max(p.x1 for p in pair)) / 2
        fig.text(xc, 0.052, xlabel, ha="center", va="center", fontsize=9.5, color=c_ink)
    fig.text(0.022, 0.455, ylabel, ha="center", va="center", rotation=90,
             fontsize=12.5, color=c_ink)
    return fig

## Figure 2: selected sites

`draw_site_panel` puts one complex on an existing axis. `build_site_panel` wraps it in
its own figure for a single plot, and `build_site_figure` lays several out in a row on
a shared axis range. The site can be an index 0 to 9 or a label like `N52K`.

In [ ]:
#Observed vs predicted for one complex, split by germline or mutant at one site
def draw_site_panel(ax, ab, ag, site, lo=None, hi=None, show_ylabel=True):
    k = site_index(ab, site)
    lab = site_labels(ab)[k]
    d = load_complex(ab, ag)
    if lo is None or hi is None:
        lo, hi = shared_range([d])
        lo = min(lo, floor - jitter_band - 0.04)

    style_axes(ax, lo, hi)
    ax.axvline(floor, color=c_line, lw=0.8, ls=(0, (4, 3)), zorder=1)
    ax.axhline(floor, color=c_line, lw=0.8, ls=(0, (4, 3)), zorder=1)

    x, y = jitter_censored(d["obs"], d["pred"], np.random.default_rng(jitter_seed))
    mut = np.array([g[k] == "M" for g in d["geno"]])
    ax.scatter(x[~mut], y[~mut], s=10, c=germ_hex, alpha=0.6, linewidths=0, zorder=2)
    ax.scatter(x[mut], y[mut], s=10, c=mut_hex, alpha=0.6, linewidths=0, zorder=3)

    ax.set_title(f"Antibody: {ab}\nAntigen: {ag}", fontsize=9, color=c_ink, pad=6)
    ax.set_xlabel(xlabel, fontsize=8, color=c_ink)
    if show_ylabel:
        ax.set_ylabel(ylabel, fontsize=10, color=c_ink, labelpad=10)
    ax.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=germ_hex, mec="none",
                              ms=6, label=f"\u2212{lab}"),
                       Line2D([0], [0], marker="o", ls="", mfc=mut_hex, mec="none",
                              ms=6, label=f"+{lab}")],
              loc="upper left", fontsize=7.5, frameon=True, framealpha=0.9,
              edgecolor=c_line, handletextpad=0.4, borderpad=0.5)
    return ax

# One standalone selected-site figure, for any antibody, antigen, and site
def build_site_panel(ab, ag, site):
    fig = plt.figure(figsize=(4.2, 4.4))
    draw_site_panel(fig.add_subplot(111), ab, ag, site)
    fig.subplots_adjust(left=0.20, right=0.96, top=0.82, bottom=0.16)
    return fig

# Row of selected-site panels sharing one axis range, as used in the supplement
def build_site_figure(specs):
    data = [load_complex(ab, ag) for ab, ag, _ in specs]
    lo, hi = shared_range(data)
    lo = min(lo, floor - jitter_band - 0.04)

    fig = plt.figure(figsize=(3.3 * len(specs), 4.1))
    gs = gridspec.GridSpec(1, len(specs), figure=fig, wspace=0.30,
                           left=0.10, right=0.99, top=0.78, bottom=0.17)
    for j, (ab, ag, site) in enumerate(specs):
        draw_site_panel(fig.add_subplot(gs[0, j]), ab, ag, site, lo, hi,
                        show_ylabel=(j == 0))
    fig.text(0.5, 0.955, "Observed vs. predicted fits at selected sites",
             ha="center", fontsize=12.5, color=c_ink, weight="bold")
    return fig

## Build the supplement figures

In [ ]:
save_figure(build_grid(), "obs_pred_full")
save_figure(build_site_figure(site_specs), "obs_pred_selected")

## One panel, any antibody and site

Pass a label or an index. `site_labels` lists what is available for an antibody.

In [ ]:
print(site_labels("1NC9"))
build_site_panel("1NC9", "CH505TF", "S56R")
plt.show()